# Notebook 0 — Model Setup

Load `meta-llama/Meta-Llama-3-8B` via TransformerLens and verify it is usable before any layer calibration or extraction begins.

**Outputs**
- Console: `d_model`, `n_layers`, estimated full activation cache size, available VRAM
- Console: sample forward pass logits on three SNOMED concept names (sanity check)

**No files written** — configuration is already recorded in `src/config.py`.

In [ ]:
# parameters
DATA_DIR = "../../data/stage1"

In [ ]:
import sys
sys.path.insert(0, "../..")

from src.config import MODEL_NAME
print(f"Model: {MODEL_NAME}")

## 0a. Load model

In [ ]:
import torch
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(MODEL_NAME)
model.eval()
print("Model loaded.")

## 0b. Memory and compatibility check

Estimates the size of a full activation cache (all layers, all concepts, all token positions) to surface any memory risk before calibration begins.

Stage 1 only needs activations at a single layer (`L_det`), so the full cache will not actually be stored — but this estimate confirms the model fits.

In [ ]:
d_model = model.cfg.d_model
n_layers = model.cfg.n_layers
n_concepts = 1879
max_tokens_per_concept = 50  # estimated upper bound for breast SNOMED terms

cache_bytes = n_concepts * max_tokens_per_concept * n_layers * d_model * 2  # fp16

print(f"d_model:        {d_model}")
print(f"n_layers:       {n_layers}")
print(f"n_concepts:     {n_concepts}")
print(f"Estimated full cache: {cache_bytes / 1e9:.1f} GB (fp16, all layers)")

if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Available VRAM: {vram:.1f} GB")
    if cache_bytes / 1e9 > vram:
        print("WARNING: full cache exceeds VRAM — single-layer extraction required (expected).")
    else:
        print("Full cache fits in VRAM.")
else:
    print("No CUDA device detected — running on CPU.")

## 0c. Forward pass sanity check

Run the model on three representative SNOMED breast concept names and print the top-5 predicted tokens. Confirms the model is loaded correctly and tokenisation is working.

In [ ]:
sample_concepts = [
    "invasive ductal carcinoma of breast",
    "malignant neoplasm of upper outer quadrant of female breast",
    "ductal carcinoma in situ of breast",
]

for concept in sample_concepts:
    tokens = model.to_tokens(concept)
    n_tokens = tokens.shape[1]
    with torch.no_grad():
        logits = model(tokens)  # (1, n_tokens, vocab_size)
    top5 = logits[0, -1].topk(5).indices
    top5_str = [model.to_string(t.unsqueeze(0)) for t in top5]
    print(f"Concept ({n_tokens} tokens): {concept!r}")
    print(f"  Top-5 next tokens: {top5_str}")
    print()